In [0]:
%run ../config/config

In [0]:
import json
import time
import requests

from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import (
    RecordType,
    StreamConfigurationOptions,
    TableProperties
)

TABLE_NAME = f"{CATALOG}.{STAGING_SCHEMA}.{ZEROBUS_STAGING}"

CLIENT_ID = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-client-id"
)

CLIENT_SECRET = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-client-secret"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME}
    (
        event_id STRING,
        source_updated BIGINT,
        ingested_at BIGINT,
        event_json STRING
    )
    USING DELTA;
""")

spark.sql(f"""
    GRANT USE CATALOG
    ON CATALOG `{CATALOG}`
    TO `{CLIENT_ID}`;      
""")

spark.sql(f"""
    GRANT USE SCHEMA
    ON SCHEMA {CATALOG}.{STAGING_SCHEMA}
    TO `{CLIENT_ID}`;    
""")

spark.sql(f"""
    GRANT SELECT, MODIFY
    ON TABLE {TABLE_NAME}
    TO `{CLIENT_ID}`;      
""")

sdk = ZerobusSdk(
    ZEROBUS_ENDPOINT,
    WORKSPACE
)

table_properties = TableProperties(TABLE_NAME)

options = StreamConfigurationOptions(
    record_type=RecordType.JSON
)

stream = sdk.create_stream(
    CLIENT_ID,
    CLIENT_SECRET,
    table_properties,
    options
)

try:
    while True:
        response = requests.get(STREAM_URL, timeout=30)
        response.raise_for_status()

        data = response.json()

        last_offset = None
        sent_count = 0

        for earthquake in data["features"]:
            record = {
                "event_id": earthquake["id"],
                "source_updated": earthquake["properties"]["updated"],
                "ingested_at": int(time.time() * 1000),
                "event_json": json.dumps(earthquake)
            }

            last_offset = stream.ingest_record_offset(record)
            sent_count += 1

        if last_offset is not None:
            stream.wait_for_offset(last_offset)

        print(f"Submitted {sent_count} records; batch acknowledged")

        time.sleep(60)
finally:
    stream.close()